<a href="https://colab.research.google.com/github/melonmeg/RAG_Medical_Assistant/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai biopython langchain-community langchain-huggingface sentence-transformers faiss-cpu openai langchain_groq sentence-transformers rank_bm25

In [ ]:
import re
import time

from openai import OpenAI
from langchain_groq import ChatGroq
from Bio import Entrez

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings




In [ ]:
import os
from sentence_transformers import CrossEncoder
os.environ["GROQ_API_KEY"]   = input("enter GROQ API KEY") # for final answer

Entrez.email = "abc@email.com"


embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},  # change to "cuda" if using a GPU
    encode_kwargs={"normalize_embeddings": True},
)

global_vectorstore = None   # NEW: persists across questions
seen_pmids = set()          # NEW: track PMIDs already indexed

llm2=ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.6,  # Groq recommends 0.5–0.7 for this model
)

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.6,  # Groq recommends 0.5–0.7 for this model
)



reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

print("✅ Setup complete")

In [ ]:
def planner_agent(question: str, conversation_history: list) -> bool:
    """Returns True if existing FAISS is sufficient, False if new evidence needed."""
    if global_vectorstore is None:
        return False

    # Sample what's already in FAISS
    sample_docs = global_vectorstore.similarity_search(question, k=5)
    sample_context = "\n\n".join(doc.page_content for doc in sample_docs)

    prompt = f"""You are a medical research planner.

A user asked: {question}

Here is a sample of what is currently indexed in the knowledge base:
{sample_context}

Conversation so far: {conversation_history}

Decide: is the existing indexed evidence sufficient to answer this question well, or do we need to search PubMed for new articles?
Consider whether the indexed evidence covers this question in sufficient depth, not just the same topic superficially.

Reply with ONLY one word: SUFFICIENT or FETCH"""

    response = llm.invoke(prompt)
    decision = response.content.strip().upper()
    print(f"✅ Planner Agent decision: {decision}")
    return decision == "SUFFICIENT"

In [ ]:
def generate_pubmed_queries(patient_history: str, question: str) -> list[str]:
    prompt = f"""You are a medical research assistant. Given a patient history and clinical question,
generate 4 optimized PubMed search queries to retrieve relevant literature.

Patient History: {patient_history}
Clinical Question: {question}

Return ONLY a numbered list of queries, one per line. No explanation."""


    response = llm.invoke(prompt)

    queries = response.content.split("\n")

    print(f"✅ Module 1: Generated {len(queries)} queries")
    for i, q in enumerate(queries, 1):
        print(f"  {i}. {q}")
    return queries

In [ ]:
def search_pubmed(query: str, max_results: int = 25) -> list[str]:
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

In [ ]:
def fetch_abstracts(pmids: list[str]) -> list[Document]:
    if not pmids:
        return []
    handle = Entrez.efetch(db="pubmed", id=",".join(pmids),
                           rettype="abstract", retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    docs = []
    for article in records["PubmedArticle"]:
        try:
            medline = article["MedlineCitation"]
            art     = medline["Article"]
            pmid    = str(medline["PMID"])
            title   = str(art.get("ArticleTitle", ""))
            abstract_texts = art.get("Abstract", {}).get("AbstractText", [])
            abstract = " ".join(str(a) for a in abstract_texts) if abstract_texts else ""
            if abstract:
                docs.append(Document(
                    page_content=f"{title}\n\n{abstract}",
                    metadata={"pmid": pmid, "title": title}
                ))
        except Exception:
            continue
    return docs


In [ ]:
def retrieve_articles(queries: list[str]) -> list[Document]:
    all_pmids, seen = [], set()
    for q in queries:
        for pmid in search_pubmed(q):
            if pmid not in seen and pmid not in seen_pmids:   # CHANGED: skip already-indexed
                seen.add(pmid)
                all_pmids.append(pmid)
        time.sleep(0.4)

    docs = fetch_abstracts(all_pmids)
    seen_pmids.update(all_pmids)   # NEW
    print(f"✅ Module 2: Retrieved {len(docs)} NEW unique articles with abstracts")
    return docs

In [ ]:
def chunk_documents(docs: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " "]
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ Module 3: Created {len(chunks)} chunks")
    return chunks

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

def hybrid_retrieve(chunks, queries, top_k=20):
    global global_vectorstore
    if chunks:   # only add if there are new chunks
        if global_vectorstore is None:
            global_vectorstore = FAISS.from_documents(chunks, embedding_model)
        else:
            global_vectorstore.add_documents(chunks)   # CHANGED: incremental add, not rebuild

    bm25 = BM25Retriever.from_documents(chunks) if chunks else None  # NOTE: BM25 has no native FAISS persistence; see note below

    seen = set()
    retrieved = []
    for query in queries:
        dense_docs = global_vectorstore.similarity_search(query, k=top_k)   # CHANGED: use global_vectorstore
        bm25_docs = bm25.invoke(query) if bm25 else []
        for doc in bm25_docs + dense_docs:
            key = doc.page_content
            if key not in seen:
                seen.add(key)
                retrieved.append(doc)

    print(f"✅ Module 4: Hybrid retrieval → {len(retrieved)} unique chunks")
    return retrieved[:top_k]

In [ ]:
def rerank_documents(query: str,
                     documents: list[Document],
                     top_k: int = 5) -> list[Document]:

    pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [doc for doc, _ in ranked[:top_k]]

    print(f"✅ Module 5: Reranked to {len(top_docs)} chunks")

    return top_docs

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are an evidence-based clinical research assistant.

History:
{history}

Retrieved Evidence:
{context}

Clinical Question:
{question}

Your task is to answer the clinical question using the retrieved evidence as the PRIMARY source of information.

Instructions:

1. Directly answer the clinical question.
2. Base recommendations primarily on the retrieved evidence.
3. If the evidence is incomplete, clearly state any assumptions or uncertainties.
4. Summarize the key findings from the evidence.
5. Compare treatment options when appropriate.
6. Mention contraindications, risks, and monitoring requirements supported by the evidence.
7. Mention conflicting findings if they exist.
8. Assign an overall confidence level (High/Moderate/Low) and briefly explain why.
9. Do not fabricate study results or citations.

Format your response with clear headings.
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

def ask(question: str, documents: list[Document]):
    context = "\n\n".join(doc.page_content for doc in documents)

    chain = prompt | llm | output_parser

    return chain.invoke({"history": "", "context": context, "question": question})

In [ ]:
print("=" * 55)
print("  MEDICAL RAG PIPELINE")
print("=" * 55)

patient_history = """
68-year-old male with type 2 diabetes (HbA1c 8.9%), hypertension, and CKD stage 3.
Currently on metformin 1000mg BD, amlodipine 5mg. eGFR: 38 mL/min.
"""

question = "What is the safest antidiabetic regimen for this patient given CKD stage 3?"

query = f"""
Patient History:
{patient_history}

Question:
{question}
"""
queries = generate_pubmed_queries(patient_history, question)


In [ ]:
if planner_agent(question, queries):
    chunks = []   # DELETE: no new download/chunking needed
else:
    articles = retrieve_articles(queries)
    chunks = chunk_documents(articles)

retrieved_docs = hybrid_retrieve(chunks, queries)   # works whether chunks is empty or new

In [ ]:
top_docs = rerank_documents(question, retrieved_docs)

In [ ]:
answer = ask(question, top_docs)

In [ ]:
print("\n" + "=" * 55)
print("  FINAL ANSWER")
print("=" * 55)
print(answer)

In [ ]:
judge_prompt = """
You are an expert medical evaluator.

Patient History:
{history}

Question:
{question}

Retrieved Medical Evidence:
{context}

AI Answer:
{answer}

Evaluate the answer using ONLY the retrieved evidence.

Give a score from 1-10 for:

1. Accuracy
2. Faithfulness to the evidence
3. Completeness
4. Clinical Safety

Finally output:

Overall Score: X/10

Reasoning:
<brief explanation>
"""

In [ ]:
def judge_answer(history, question, context_docs, answer):
    context = "\n\n".join(
        doc.page_content for doc in context_docs
    )

    prompt = judge_prompt.format(
        history=history,
        question=question,
        context=context,
        answer=answer
    )

    response = llm2.invoke(prompt)

    return response.content

In [ ]:
answer = ask(question, top_docs)

evaluation = judge_answer(
    patient_history,
    question,
    top_docs,
    answer
)

print(evaluation)

In [ ]:
# ============================================================
#  PLANNER AGENT DEEP TESTS
# ============================================================

# First, build a FAISS index with diabetes/CKD content
setup_history = "68-year-old male with type 2 diabetes, CKD stage 3, eGFR 38."
setup_question = "What is the safest antidiabetic regimen for CKD stage 3?"
setup_queries = generate_pubmed_queries(setup_history, setup_question)
setup_articles = retrieve_articles(setup_queries)
setup_chunks = chunk_documents(setup_articles)
hybrid_retrieve(setup_chunks, setup_queries)  # populates global_vectorstore
print(f"\nFAISS built with {len(setup_chunks)} chunks on diabetes/CKD topic\n")
print("=" * 50)

planner_results = []

def log_planner(name, decision, expected, detail=""):
    correct = decision == expected
    status = "✅" if correct else "⚠️ UNEXPECTED"
    expected_str = "SUFFICIENT" if expected else "FETCH"
    got_str = "SUFFICIENT" if decision else "FETCH"
    planner_results.append((name, correct, got_str))
    print(f"{status} | {name}")
    print(f"         expected={expected_str}, got={got_str}" + (f" | {detail}" if detail else ""))
    print()

# ── Test 1: Same topic, same question ────────────────────────
# Should be SUFFICIENT — exact topic already indexed
decision = planner_agent(
    "What is the safest antidiabetic regimen for CKD stage 3?",
    []
)
log_planner("Same topic, same question", decision, expected=True)

# ── Test 2: Same topic, follow-up question ────────────────────
# Should be SUFFICIENT — close enough to indexed content
decision = planner_agent(
    "Can SGLT2 inhibitors be used in CKD stage 3b?",
    [("What is the safest antidiabetic regimen for CKD stage 3?", "SGLT2 inhibitors are preferred...")]
)
log_planner("Same topic, follow-up question", decision, expected=True)

# ── Test 3: Slightly different but related ────────────────────
# Borderline — LLM's judgment matters here
decision = planner_agent(
    "What are the cardiovascular risks of GLP-1 agonists in diabetic nephropathy?",
    []
)
log_planner("Related but different angle (borderline)", decision, expected=True,
            detail="acceptable either way — tests LLM reasoning")

# ── Test 4: Completely different topic ────────────────────────
# Should FETCH — nothing about this in FAISS
decision = planner_agent(
    "What is the first-line treatment for community acquired pneumonia?",
    []
)
log_planner("Completely different topic", decision, expected=False)

# ── Test 5: Different specialty ───────────────────────────────
# Should FETCH — cardiology, not nephrology/diabetes
decision = planner_agent(
    "When should we initiate anticoagulation after atrial fibrillation diagnosis?",
    []
)
log_planner("Different specialty (cardiology)", decision, expected=False)

# ── Test 6: Conversation history steers decision ─────────────
# History shows we've been discussing pneumonia → should FETCH
decision = planner_agent(
    "What is the recommended antibiotic duration?",   # vague — history gives context
    [
        ("What causes community acquired pneumonia?", "S. pneumoniae is the most common..."),
        ("What antibiotics are first line?", "Amoxicillin is recommended...")
    ]
)
log_planner("Vague question, off-topic history → FETCH", decision, expected=False)

# ── Test 7: Conversation history on same topic ────────────────
# History is about diabetes/CKD → should be SUFFICIENT
decision = planner_agent(
    "What about dosing adjustments for metformin?",   # vague without history
    [
        ("What is the safest antidiabetic regimen for CKD stage 3?", "Metformin should be used cautiously..."),
    ]
)
log_planner("Vague question, on-topic history → SUFFICIENT", decision, expected=True)

# ── Test 8: Adversarial — misleading keywords ─────────────────
# Contains "diabetes" but is really about surgery — should FETCH
decision = planner_agent(
    "What are perioperative glucose management protocols for diabetic patients undergoing cardiac surgery?",
    []
)
log_planner("Adversarial: diabetes keyword but different context", decision, expected=False,
            detail="tests whether LLM looks beyond surface keywords")

# ── Summary ───────────────────────────────────────────────────
print("=" * 50)
print("  PLANNER TEST SUMMARY")
print("=" * 50)
correct = sum(1 for _, c, _ in planner_results if c)
print(f"  {correct}/{len(planner_results)} matched expected\n")
for name, c, got in planner_results:
    icon = "✅" if c else "⚠️"
    print(f"  {icon} [{got:>10}]  {name}")
print()
print("Note: ⚠️ doesn't always mean wrong — borderline cases")
print("      are judgment calls. Review the LLM's reasoning.")
print("=" * 50)

In [ ]:
import gradio as gr

conversation_history = []  # ADD THIS above the function, global state

def respond(age, sex, weight, chief_complaint, symptom_duration,
            medical_history, medications, allergies, vitals, lab_results,
            question, chat_history):
    try:
        patient_history = f"""
Age: {age}, Sex: {sex}, Weight: {weight}
Chief Complaint: {chief_complaint}
Duration: {symptom_duration}
Medical History: {medical_history}
Current Medications: {medications}
Allergies: {allergies}
Vitals: {vitals}
Labs: {lab_results}
"""
        # 1. Planner decides
        queries = generate_pubmed_queries(patient_history, question)

        if planner_agent(question, conversation_history):
            # FAISS sufficient — skip fetch
            chunks = []
        else:
            # Fetch new evidence
            articles = retrieve_articles(queries)
            chunks = chunk_documents(articles)

        # 2. Hybrid retrieve from (updated) FAISS
        retrieved_docs = hybrid_retrieve(chunks, queries)

        # 3. Rerank
        top_docs = rerank_documents(question, retrieved_docs)

        # 4. Build history string for prompt
        history_text = "\n".join(
            f"Q: {q}\nA: {a}" for q, a in conversation_history
        )

        # 5. Answer
        context = "\n\n".join(doc.page_content for doc in top_docs)
        chain = prompt | llm | StrOutputParser()
        response = chain.invoke({
            "history": history_text,
            "context": context,
            "question": question
        })

        # 6. Update history
        conversation_history.append((question, response))
        chat_history.append((question, response))

        return "", chat_history

    except Exception as e:
        import traceback
        traceback.print_exc()
        raise e

with gr.Blocks(title="Clinical Research Assistant") as demo:

    gr.Markdown("# 🩺 Clinical Research Assistant")
    gr.Markdown("Enter the patient's clinical information before asking your question.")

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("## Patient History")

            age = gr.Number(
                label="Age",
                precision=0
            )

            sex = gr.Dropdown(
                ["Male", "Female", "Other"],
                label="Sex"
            )

            weight = gr.Number(
                label="Weight (kg)"
            )

            chief_complaint = gr.Textbox(
                label="Chief Complaint",
                lines=2,
                placeholder="e.g. Chest pain, fever, cough..."
            )

            symptom_duration = gr.Textbox(
                label="Duration / Onset",
                placeholder="e.g. Started 3 days ago"
            )

            medical_history = gr.Textbox(
                label="Past Medical History",
                lines=4,
                placeholder="Hypertension, Type 2 Diabetes..."
            )

            medications = gr.Textbox(
                label="Current Medications",
                lines=3,
                placeholder="Metformin, Aspirin..."
            )

            allergies = gr.Textbox(
                label="Drug Allergies",
                placeholder="Penicillin"
            )

            vitals = gr.Textbox(
                label="Vital Signs",
                lines=3,
                placeholder="""BP: 130/80
HR: 72
Temp: 98.6°F
SpO₂: 98%"""
            )

            lab_results = gr.Textbox(
                label="Laboratory Results",
                lines=4,
                placeholder="""HbA1c: 8.2%
Creatinine: 1.3
eGFR: 62"""
            )

        with gr.Column(scale=2):

            chatbot = gr.Chatbot(
                label="Clinical Assistant",
                height=550
            )

            question = gr.Textbox(
                label="Clinical Question",
                placeholder="Ask a question about this patient..."
            )

            submit = gr.Button(
                "Generate Evidence-Based Recommendation",
                variant="primary"
            )

    submit.click(
        respond,
        inputs=[
            age,
            sex,
            weight,
            chief_complaint,
            symptom_duration,
            medical_history,
            medications,
            allergies,
            vitals,
            lab_results,
            question,
            chatbot
        ],
        outputs=[
            question,
            chatbot
        ]
    )

    question.submit(
        respond,
        inputs=[
            age,
            sex,
            weight,
            chief_complaint,
            symptom_duration,
            medical_history,
            medications,
            allergies,
            vitals,
            lab_results,
            question,
            chatbot
        ],
        outputs=[
            question,
            chatbot
        ]
    )

demo.launch(share=True)